# Evaluation of HuBERT on our synthetic generated data

In [2]:
import os
import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import HubertModel
from tqdm import tqdm
import numpy as np

# === CONFIG ===
class Config:
    SAMPLE_RATE = 16000
    MODEL_NAME = "facebook/hubert-base-ls960"
    BATCH_SIZE = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_DIR = "data/trimmed_fan"
    MODEL_PATH = "models/hubert_transformer_synthetic.pth"

config = Config()

# === MODEL DEFINITION ===
class HubertClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2):
        super(HubertClassifier, self).__init__()
        self.hubert = HubertModel.from_pretrained(config.MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_values):
        with torch.no_grad():
            features = self.hubert(input_values).last_hidden_state
        pooled = torch.mean(features, dim=1)
        logits = self.classifier(pooled)
        return logits

# === ENHANCED AUDIO LOADING ===
def load_audio(path):
    """Enhanced audio loading with multiple fallbacks"""
    try:
        waveform, sr = torchaudio.load(path)
    except Exception as e:
        try:
            # Try explicitly with soundfile backend
            torchaudio.set_audio_backend("soundfile")
            waveform, sr = torchaudio.load(path)
        except Exception as e2:
            try:
                # Fallback to librosa if available
                import librosa
                y, sr = librosa.load(path, sr=None, mono=True)
                waveform = torch.tensor(y).unsqueeze(0)
            except ImportError:
                # Last resort - read with scipy
                from scipy.io import wavfile
                sr, y = wavfile.read(path)
                if y.ndim > 1:
                    y = np.mean(y, axis=1)
                # Convert to float and normalize if needed
                if y.dtype == np.int16:
                    y = y.astype(np.float32) / 32768.0
                elif y.dtype == np.int32:
                    y = y.astype(np.float32) / 2147483648.0
                waveform = torch.tensor(y).unsqueeze(0)
    
    # Make mono if stereo
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # Resample if needed
    if sr != config.SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(sr, config.SAMPLE_RATE)
        waveform = resampler(waveform)
    
    return waveform

# === DATASET FOR RAW AUDIO ===
class AudioDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        self.label_map = {"normal": 0, "abnormal": 1}

        for label_str in self.label_map.keys():
            label = self.label_map[label_str]
            class_dir = os.path.join(root_dir, label_str)
            for root, _, files in os.walk(class_dir):
                for file in files:
                    if file.endswith(".wav"):
                        self.samples.append((os.path.join(root, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        try:
            waveform = load_audio(file_path)
            return waveform.squeeze(0), label
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            # Return a placeholder or skip this sample
            # For simplicity, we'll return a zero tensor with the correct shape
            return torch.zeros(config.SAMPLE_RATE), label

# === COLLATE FUNCTION ===
def collate_fn(batch):
    """Custom collate function to handle variable length audio"""
    waveforms = []
    labels = []
    
    for waveform, label in batch:
        # Ensure minimum length (1 second)
        if waveform.shape[0] < config.SAMPLE_RATE:
            # Pad if too short
            padding = torch.zeros(config.SAMPLE_RATE - waveform.shape[0])
            waveform = torch.cat([waveform, padding])
        
        # Trim if too long (for batch processing)
        # HuBERT can handle longer sequences, but we'll limit for memory efficiency
        max_length = 10 * config.SAMPLE_RATE  # 10 seconds maximum
        if waveform.shape[0] > max_length:
            waveform = waveform[:max_length]
            
        waveforms.append(waveform)
        labels.append(label)
    
    # Pad all waveforms to the same length
    max_len = max(wav.shape[0] for wav in waveforms)
    padded_waveforms = []
    
    for waveform in waveforms:
        if waveform.shape[0] < max_len:
            padding = torch.zeros(max_len - waveform.shape[0])
            padded_waveform = torch.cat([waveform, padding])
        else:
            padded_waveform = waveform
        padded_waveforms.append(padded_waveform)
    
    # Stack into a batch
    waveforms_tensor = torch.stack(padded_waveforms)
    labels_tensor = torch.tensor(labels)
    
    return waveforms_tensor, labels_tensor

# === MAIN EVALUATION FUNCTION ===
def main():
    # Load the model
    model = HubertClassifier().to(config.DEVICE)
    model.load_state_dict(torch.load(config.MODEL_PATH, map_location=config.DEVICE))
    model.eval()
    
    # Create dataset and dataloader
    dataset = AudioDataset(config.DATA_DIR)
    dataloader = DataLoader(
        dataset, 
        batch_size=config.BATCH_SIZE, 
        shuffle=False,
        collate_fn=collate_fn
    )
    
    total = 0
    correct = 0
    predicted_normal = 0
    predicted_abnormal = 0
    
    # Track true positive, false positive, true negative, false negative
    tp = 0  # Predicted abnormal when truly abnormal
    fp = 0  # Predicted abnormal when truly normal
    tn = 0  # Predicted normal when truly normal
    fn = 0  # Predicted normal when truly abnormal
    
    true_normal = 0
    true_abnormal = 0

    with torch.no_grad():
        for waveforms, labels in tqdm(dataloader, desc="Evaluating"):
            waveforms = waveforms.to(config.DEVICE)
            labels = labels.to(config.DEVICE)
            
            # Run inference on the waveform
            outputs = model(waveforms)
            preds = torch.argmax(outputs, dim=1)
            
            # Update counts
            predicted_normal += (preds == 0).sum().item()
            predicted_abnormal += (preds == 1).sum().item()
            
            # Update confusion matrix
            batch_tp = ((preds == 1) & (labels == 1)).sum().item()
            batch_fp = ((preds == 1) & (labels == 0)).sum().item()
            batch_tn = ((preds == 0) & (labels == 0)).sum().item()
            batch_fn = ((preds == 0) & (labels == 1)).sum().item()
            
            tp += batch_tp
            fp += batch_fp
            tn += batch_tn
            fn += batch_fn
            
            # Update true counts
            true_normal += (labels == 0).sum().item()
            true_abnormal += (labels == 1).sum().item()
            
            # Update total correct
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100.0 * correct / total if total > 0 else 0.0
    
    # Calculate precision, recall and F1 score
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Print results
    print("\n===== EVALUATION RESULTS =====")
    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total})")
    print(f"\nPREDICTION COUNTS:")
    print(f"  - Predicted Normal (0): {predicted_normal}")
    print(f"  - Predicted Abnormal (1): {predicted_abnormal}")
    print(f"\nACTUAL COUNTS:")
    print(f"  - Actually Normal (0): {true_normal}")
    print(f"  - Actually Abnormal (1): {true_abnormal}")
    print(f"\nMETRICS:")
    print(f"  - Precision: {precision:.4f}")
    print(f"  - Recall: {recall:.4f}")
    print(f"  - F1 Score: {f1:.4f}")

if __name__ == "__main__":
    main()

/var/folders/m9/460mzb1j78b5x0bdfn30jtxm0000gn/T/ipykernel_95241/1614434436.py:152: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(config.MOD


===== EVALUATION RESULTS =====
Accuracy: 48.75% (117/240)

PREDICTION COUNTS:
  - Predicted Normal (0): 59
  - Predicted Abnormal (1): 181

ACTUAL COUNTS:
  - Actually Normal (0): 120
  - Actually Abnormal (1): 120

METRICS:
  - Precision: 0.4917
  - Recall: 0.7417
  - F1 Score: 0.5914


In [3]:
import os
import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import HubertModel
from tqdm import tqdm
import numpy as np

# === CONFIG ===
class Config:
    SAMPLE_RATE = 16000
    MODEL_NAME = "facebook/hubert-base-ls960"
    BATCH_SIZE = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_DIR = "data/trimmed_fan"
    MODEL_PATH = "hubert_transformer_trimmed_fan.pth"

config = Config()

# === MODEL DEFINITION ===
class HubertClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2):
        super(HubertClassifier, self).__init__()
        self.hubert = HubertModel.from_pretrained(config.MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_values):
        with torch.no_grad():
            features = self.hubert(input_values).last_hidden_state
        pooled = torch.mean(features, dim=1)
        logits = self.classifier(pooled)
        return logits

# === ENHANCED AUDIO LOADING ===
def load_audio(path):
    """Enhanced audio loading with multiple fallbacks"""
    try:
        waveform, sr = torchaudio.load(path)
    except Exception as e:
        try:
            # Try explicitly with soundfile backend
            torchaudio.set_audio_backend("soundfile")
            waveform, sr = torchaudio.load(path)
        except Exception as e2:
            try:
                # Fallback to librosa if available
                import librosa
                y, sr = librosa.load(path, sr=None, mono=True)
                waveform = torch.tensor(y).unsqueeze(0)
            except ImportError:
                # Last resort - read with scipy
                from scipy.io import wavfile
                sr, y = wavfile.read(path)
                if y.ndim > 1:
                    y = np.mean(y, axis=1)
                # Convert to float and normalize if needed
                if y.dtype == np.int16:
                    y = y.astype(np.float32) / 32768.0
                elif y.dtype == np.int32:
                    y = y.astype(np.float32) / 2147483648.0
                waveform = torch.tensor(y).unsqueeze(0)
    
    # Make mono if stereo
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # Resample if needed
    if sr != config.SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(sr, config.SAMPLE_RATE)
        waveform = resampler(waveform)
    
    return waveform

# === DATASET FOR RAW AUDIO ===
class AudioDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        self.label_map = {"normal": 0, "abnormal": 1}

        for label_str in self.label_map.keys():
            label = self.label_map[label_str]
            class_dir = os.path.join(root_dir, label_str)
            for root, _, files in os.walk(class_dir):
                for file in files:
                    if file.endswith(".wav"):
                        self.samples.append((os.path.join(root, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        try:
            waveform = load_audio(file_path)
            return waveform.squeeze(0), label
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            # Return a placeholder or skip this sample
            # For simplicity, we'll return a zero tensor with the correct shape
            return torch.zeros(config.SAMPLE_RATE), label

# === COLLATE FUNCTION ===
def collate_fn(batch):
    """Custom collate function to handle variable length audio"""
    waveforms = []
    labels = []
    
    for waveform, label in batch:
        # Ensure minimum length (1 second)
        if waveform.shape[0] < config.SAMPLE_RATE:
            # Pad if too short
            padding = torch.zeros(config.SAMPLE_RATE - waveform.shape[0])
            waveform = torch.cat([waveform, padding])
        
        # Trim if too long (for batch processing)
        # HuBERT can handle longer sequences, but we'll limit for memory efficiency
        max_length = 10 * config.SAMPLE_RATE  # 10 seconds maximum
        if waveform.shape[0] > max_length:
            waveform = waveform[:max_length]
            
        waveforms.append(waveform)
        labels.append(label)
    
    # Pad all waveforms to the same length
    max_len = max(wav.shape[0] for wav in waveforms)
    padded_waveforms = []
    
    for waveform in waveforms:
        if waveform.shape[0] < max_len:
            padding = torch.zeros(max_len - waveform.shape[0])
            padded_waveform = torch.cat([waveform, padding])
        else:
            padded_waveform = waveform
        padded_waveforms.append(padded_waveform)
    
    # Stack into a batch
    waveforms_tensor = torch.stack(padded_waveforms)
    labels_tensor = torch.tensor(labels)
    
    return waveforms_tensor, labels_tensor

# === MAIN EVALUATION FUNCTION ===
def main():
    # Load the model
    model = HubertClassifier().to(config.DEVICE)
    model.load_state_dict(torch.load(config.MODEL_PATH, map_location=config.DEVICE))
    model.eval()
    
    # Create dataset and dataloader
    dataset = AudioDataset(config.DATA_DIR)
    dataloader = DataLoader(
        dataset, 
        batch_size=config.BATCH_SIZE, 
        shuffle=False,
        collate_fn=collate_fn
    )
    
    total = 0
    correct = 0
    predicted_normal = 0
    predicted_abnormal = 0
    
    # Track true positive, false positive, true negative, false negative
    tp = 0  # Predicted abnormal when truly abnormal
    fp = 0  # Predicted abnormal when truly normal
    tn = 0  # Predicted normal when truly normal
    fn = 0  # Predicted normal when truly abnormal
    
    true_normal = 0
    true_abnormal = 0

    with torch.no_grad():
        for waveforms, labels in tqdm(dataloader, desc="Evaluating"):
            waveforms = waveforms.to(config.DEVICE)
            labels = labels.to(config.DEVICE)
            
            # Run inference on the waveform
            outputs = model(waveforms)
            preds = torch.argmax(outputs, dim=1)
            
            # Update counts
            predicted_normal += (preds == 0).sum().item()
            predicted_abnormal += (preds == 1).sum().item()
            
            # Update confusion matrix
            batch_tp = ((preds == 1) & (labels == 1)).sum().item()
            batch_fp = ((preds == 1) & (labels == 0)).sum().item()
            batch_tn = ((preds == 0) & (labels == 0)).sum().item()
            batch_fn = ((preds == 0) & (labels == 1)).sum().item()
            
            tp += batch_tp
            fp += batch_fp
            tn += batch_tn
            fn += batch_fn
            
            # Update true counts
            true_normal += (labels == 0).sum().item()
            true_abnormal += (labels == 1).sum().item()
            
            # Update total correct
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100.0 * correct / total if total > 0 else 0.0
    
    # Calculate precision, recall and F1 score
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Print results
    print("\n===== EVALUATION RESULTS =====")
    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total})")
    print(f"\nPREDICTION COUNTS:")
    print(f"  - Predicted Normal (0): {predicted_normal}")
    print(f"  - Predicted Abnormal (1): {predicted_abnormal}")
    print(f"\nACTUAL COUNTS:")
    print(f"  - Actually Normal (0): {true_normal}")
    print(f"  - Actually Abnormal (1): {true_abnormal}")
    print(f"\nMETRICS:")
    print(f"  - Precision: {precision:.4f}")
    print(f"  - Recall: {recall:.4f}")
    print(f"  - F1 Score: {f1:.4f}")

if __name__ == "__main__":
    main()

/var/folders/m9/460mzb1j78b5x0bdfn30jtxm0000gn/T/ipykernel_95241/3662824523.py:152: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(config.MOD


===== EVALUATION RESULTS =====
Accuracy: 25.83% (62/240)

PREDICTION COUNTS:
  - Predicted Normal (0): 172
  - Predicted Abnormal (1): 68

ACTUAL COUNTS:
  - Actually Normal (0): 120
  - Actually Abnormal (1): 120

METRICS:
  - Precision: 0.0735
  - Recall: 0.0417
  - F1 Score: 0.0532


# Hubert Transformer with batch size of 8 and learning rate of 1e-4 on trimmed_fan

In [4]:
import os
import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import HubertModel
from tqdm import tqdm
import numpy as np

# === CONFIG ===
class Config:
    SAMPLE_RATE = 16000
    MODEL_NAME = "facebook/hubert-base-ls960"
    BATCH_SIZE = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_DIR = "data/trimmed_fan"
    MODEL_PATH = "hubert_transformer_trimmed_fan_bs8_lr4.pth"

config = Config()

# === MODEL DEFINITION ===
class HubertClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2):
        super(HubertClassifier, self).__init__()
        self.hubert = HubertModel.from_pretrained(config.MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_values):
        with torch.no_grad():
            features = self.hubert(input_values).last_hidden_state
        pooled = torch.mean(features, dim=1)
        logits = self.classifier(pooled)
        return logits

# === ENHANCED AUDIO LOADING ===
def load_audio(path):
    """Enhanced audio loading with multiple fallbacks"""
    try:
        waveform, sr = torchaudio.load(path)
    except Exception as e:
        try:
            # Try explicitly with soundfile backend
            torchaudio.set_audio_backend("soundfile")
            waveform, sr = torchaudio.load(path)
        except Exception as e2:
            try:
                # Fallback to librosa if available
                import librosa
                y, sr = librosa.load(path, sr=None, mono=True)
                waveform = torch.tensor(y).unsqueeze(0)
            except ImportError:
                # Last resort - read with scipy
                from scipy.io import wavfile
                sr, y = wavfile.read(path)
                if y.ndim > 1:
                    y = np.mean(y, axis=1)
                # Convert to float and normalize if needed
                if y.dtype == np.int16:
                    y = y.astype(np.float32) / 32768.0
                elif y.dtype == np.int32:
                    y = y.astype(np.float32) / 2147483648.0
                waveform = torch.tensor(y).unsqueeze(0)
    
    # Make mono if stereo
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # Resample if needed
    if sr != config.SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(sr, config.SAMPLE_RATE)
        waveform = resampler(waveform)
    
    return waveform

# === DATASET FOR RAW AUDIO ===
class AudioDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        self.label_map = {"normal": 0, "abnormal": 1}

        for label_str in self.label_map.keys():
            label = self.label_map[label_str]
            class_dir = os.path.join(root_dir, label_str)
            for root, _, files in os.walk(class_dir):
                for file in files:
                    if file.endswith(".wav"):
                        self.samples.append((os.path.join(root, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        try:
            waveform = load_audio(file_path)
            return waveform.squeeze(0), label
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            # Return a placeholder or skip this sample
            # For simplicity, we'll return a zero tensor with the correct shape
            return torch.zeros(config.SAMPLE_RATE), label

# === COLLATE FUNCTION ===
def collate_fn(batch):
    """Custom collate function to handle variable length audio"""
    waveforms = []
    labels = []
    
    for waveform, label in batch:
        # Ensure minimum length (1 second)
        if waveform.shape[0] < config.SAMPLE_RATE:
            # Pad if too short
            padding = torch.zeros(config.SAMPLE_RATE - waveform.shape[0])
            waveform = torch.cat([waveform, padding])
        
        # Trim if too long (for batch processing)
        # HuBERT can handle longer sequences, but we'll limit for memory efficiency
        max_length = 10 * config.SAMPLE_RATE  # 10 seconds maximum
        if waveform.shape[0] > max_length:
            waveform = waveform[:max_length]
            
        waveforms.append(waveform)
        labels.append(label)
    
    # Pad all waveforms to the same length
    max_len = max(wav.shape[0] for wav in waveforms)
    padded_waveforms = []
    
    for waveform in waveforms:
        if waveform.shape[0] < max_len:
            padding = torch.zeros(max_len - waveform.shape[0])
            padded_waveform = torch.cat([waveform, padding])
        else:
            padded_waveform = waveform
        padded_waveforms.append(padded_waveform)
    
    # Stack into a batch
    waveforms_tensor = torch.stack(padded_waveforms)
    labels_tensor = torch.tensor(labels)
    
    return waveforms_tensor, labels_tensor

# === MAIN EVALUATION FUNCTION ===
def main():
    # Load the model
    model = HubertClassifier().to(config.DEVICE)
    model.load_state_dict(torch.load(config.MODEL_PATH, map_location=config.DEVICE))
    model.eval()
    
    # Create dataset and dataloader
    dataset = AudioDataset(config.DATA_DIR)
    dataloader = DataLoader(
        dataset, 
        batch_size=config.BATCH_SIZE, 
        shuffle=False,
        collate_fn=collate_fn
    )
    
    total = 0
    correct = 0
    predicted_normal = 0
    predicted_abnormal = 0
    
    # Track true positive, false positive, true negative, false negative
    tp = 0  # Predicted abnormal when truly abnormal
    fp = 0  # Predicted abnormal when truly normal
    tn = 0  # Predicted normal when truly normal
    fn = 0  # Predicted normal when truly abnormal
    
    true_normal = 0
    true_abnormal = 0

    with torch.no_grad():
        for waveforms, labels in tqdm(dataloader, desc="Evaluating"):
            waveforms = waveforms.to(config.DEVICE)
            labels = labels.to(config.DEVICE)
            
            # Run inference on the waveform
            outputs = model(waveforms)
            preds = torch.argmax(outputs, dim=1)
            
            # Update counts
            predicted_normal += (preds == 0).sum().item()
            predicted_abnormal += (preds == 1).sum().item()
            
            # Update confusion matrix
            batch_tp = ((preds == 1) & (labels == 1)).sum().item()
            batch_fp = ((preds == 1) & (labels == 0)).sum().item()
            batch_tn = ((preds == 0) & (labels == 0)).sum().item()
            batch_fn = ((preds == 0) & (labels == 1)).sum().item()
            
            tp += batch_tp
            fp += batch_fp
            tn += batch_tn
            fn += batch_fn
            
            # Update true counts
            true_normal += (labels == 0).sum().item()
            true_abnormal += (labels == 1).sum().item()
            
            # Update total correct
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100.0 * correct / total if total > 0 else 0.0
    
    # Calculate precision, recall and F1 score
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Print results
    print("\n===== EVALUATION RESULTS =====")
    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total})")
    print(f"\nPREDICTION COUNTS:")
    print(f"  - Predicted Normal (0): {predicted_normal}")
    print(f"  - Predicted Abnormal (1): {predicted_abnormal}")
    print(f"\nACTUAL COUNTS:")
    print(f"  - Actually Normal (0): {true_normal}")
    print(f"  - Actually Abnormal (1): {true_abnormal}")
    print(f"\nMETRICS:")
    print(f"  - Precision: {precision:.4f}")
    print(f"  - Recall: {recall:.4f}")
    print(f"  - F1 Score: {f1:.4f}")

if __name__ == "__main__":
    main()

/var/folders/m9/460mzb1j78b5x0bdfn30jtxm0000gn/T/ipykernel_95241/814941369.py:152: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(config.MODE


===== EVALUATION RESULTS =====
Accuracy: 26.67% (64/240)

PREDICTION COUNTS:
  - Predicted Normal (0): 156
  - Predicted Abnormal (1): 84

ACTUAL COUNTS:
  - Actually Normal (0): 120
  - Actually Abnormal (1): 120

METRICS:
  - Precision: 0.1667
  - Recall: 0.1167
  - F1 Score: 0.1373


# HuBERT Evaluation with normalization

In [3]:
import os
import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import HubertModel
from tqdm import tqdm
import numpy as np

# === CONFIG ===
class Config:
    SAMPLE_RATE = 16000
    MODEL_NAME = "facebook/hubert-base-ls960"
    BATCH_SIZE = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_DIR = "data/trimmed_fan"
    MODEL_PATH = "hubert_transformer_trimmed_fan.pth"

config = Config()

# === MODEL DEFINITION ===
class HubertClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2):
        super(HubertClassifier, self).__init__()
        self.hubert = HubertModel.from_pretrained(config.MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_values):
        with torch.no_grad():
            features = self.hubert(input_values).last_hidden_state
        pooled = torch.mean(features, dim=1)
        logits = self.classifier(pooled)
        return logits

# === ENHANCED AUDIO LOADING ===
def load_audio(path):
    """Enhanced audio loading with multiple fallbacks"""
    try:
        waveform, sr = torchaudio.load(path)
    except Exception as e:
        try:
            # Try explicitly with soundfile backend
            torchaudio.set_audio_backend("soundfile")
            waveform, sr = torchaudio.load(path)
        except Exception as e2:
            try:
                # Fallback to librosa if available
                import librosa
                y, sr = librosa.load(path, sr=None, mono=True)
                waveform = torch.tensor(y).unsqueeze(0)
            except ImportError:
                # Last resort - read with scipy
                from scipy.io import wavfile
                sr, y = wavfile.read(path)
                if y.ndim > 1:
                    y = np.mean(y, axis=1)
                # Convert to float and normalize if needed
                if y.dtype == np.int16:
                    y = y.astype(np.float32) / 32768.0
                elif y.dtype == np.int32:
                    y = y.astype(np.float32) / 2147483648.0
                waveform = torch.tensor(y).unsqueeze(0)
    
    # Make mono if stereo
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # Resample if needed
    if sr != config.SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(sr, config.SAMPLE_RATE)
        waveform = resampler(waveform)
    
    return waveform

# === DATASET FOR RAW AUDIO ===
class AudioDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        self.label_map = {"normal": 0, "abnormal": 1}

        for label_str in self.label_map.keys():
            label = self.label_map[label_str]
            class_dir = os.path.join(root_dir, label_str)
            for root, _, files in os.walk(class_dir):
                for file in files:
                    if file.endswith(".wav"):
                        self.samples.append((os.path.join(root, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        try:
            waveform = load_audio(file_path)
            return waveform.squeeze(0), label
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            # Return a placeholder or skip this sample
            # For simplicity, we'll return a zero tensor with the correct shape
            return torch.zeros(config.SAMPLE_RATE), label

# === COLLATE FUNCTION ===
def collate_fn(batch):
    """Custom collate function to handle variable length audio"""
    waveforms = []
    labels = []
    
    for waveform, label in batch:
        # Ensure minimum length (1 second)
        if waveform.shape[0] < config.SAMPLE_RATE:
            # Pad if too short
            padding = torch.zeros(config.SAMPLE_RATE - waveform.shape[0])
            waveform = torch.cat([waveform, padding])
        
        # Trim if too long (for batch processing)
        # HuBERT can handle longer sequences, but we'll limit for memory efficiency
        max_length = 10 * config.SAMPLE_RATE  # 10 seconds maximum
        if waveform.shape[0] > max_length:
            waveform = waveform[:max_length]
            
        waveforms.append(waveform)
        labels.append(label)
    
    # Pad all waveforms to the same length
    max_len = max(wav.shape[0] for wav in waveforms)
    padded_waveforms = []
    
    for waveform in waveforms:
        if waveform.shape[0] < max_len:
            padding = torch.zeros(max_len - waveform.shape[0])
            padded_waveform = torch.cat([waveform, padding])
        else:
            padded_waveform = waveform
        padded_waveforms.append(padded_waveform)
    
    # Stack into a batch
    waveforms_tensor = torch.stack(padded_waveforms)
    labels_tensor = torch.tensor(labels)
    
    return waveforms_tensor, labels_tensor

# === MAIN EVALUATION FUNCTION ===
def main():
    # Load the model
    model = HubertClassifier().to(config.DEVICE)
    model.load_state_dict(torch.load(config.MODEL_PATH, map_location=config.DEVICE))
    model.eval()
    
    # Create dataset and dataloader
    dataset = AudioDataset(config.DATA_DIR)
    dataloader = DataLoader(
        dataset, 
        batch_size=config.BATCH_SIZE, 
        shuffle=False,
        collate_fn=collate_fn
    )
    
    total = 0
    correct = 0
    predicted_normal = 0
    predicted_abnormal = 0
    
    # Track true positive, false positive, true negative, false negative
    tp = 0  # Predicted abnormal when truly abnormal
    fp = 0  # Predicted abnormal when truly normal
    tn = 0  # Predicted normal when truly normal
    fn = 0  # Predicted normal when truly abnormal
    
    true_normal = 0
    true_abnormal = 0

    with torch.no_grad():
        for waveforms, labels in tqdm(dataloader, desc="Evaluating"):
            waveforms = waveforms.to(config.DEVICE)
            labels = labels.to(config.DEVICE)
            
            # Run inference on the waveform
            outputs = model(waveforms)
            preds = torch.argmax(outputs, dim=1)
            
            # Update counts
            predicted_normal += (preds == 0).sum().item()
            predicted_abnormal += (preds == 1).sum().item()
            
            # Update confusion matrix
            batch_tp = ((preds == 1) & (labels == 1)).sum().item()
            batch_fp = ((preds == 1) & (labels == 0)).sum().item()
            batch_tn = ((preds == 0) & (labels == 0)).sum().item()
            batch_fn = ((preds == 0) & (labels == 1)).sum().item()
            
            tp += batch_tp
            fp += batch_fp
            tn += batch_tn
            fn += batch_fn
            
            # Update true counts
            true_normal += (labels == 0).sum().item()
            true_abnormal += (labels == 1).sum().item()
            
            # Update total correct
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100.0 * correct / total if total > 0 else 0.0
    
    # Calculate precision, recall and F1 score
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Print results
    print("\n===== EVALUATION RESULTS =====")
    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total})")
    print(f"\nPREDICTION COUNTS:")
    print(f"  - Predicted Normal (0): {predicted_normal}")
    print(f"  - Predicted Abnormal (1): {predicted_abnormal}")
    print(f"\nACTUAL COUNTS:")
    print(f"  - Actually Normal (0): {true_normal}")
    print(f"  - Actually Abnormal (1): {true_abnormal}")
    print(f"\nMETRICS:")
    print(f"  - Precision: {precision:.4f}")
    print(f"  - Recall: {recall:.4f}")
    print(f"  - F1 Score: {f1:.4f}")

if __name__ == "__main__":
    main()

/var/folders/m9/460mzb1j78b5x0bdfn30jtxm0000gn/T/ipykernel_98164/3662824523.py:152: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(config.MOD


===== EVALUATION RESULTS =====
Accuracy: 25.83% (62/240)

PREDICTION COUNTS:
  - Predicted Normal (0): 172
  - Predicted Abnormal (1): 68

ACTUAL COUNTS:
  - Actually Normal (0): 120
  - Actually Abnormal (1): 120

METRICS:
  - Precision: 0.0735
  - Recall: 0.0417
  - F1 Score: 0.0532
